# DNA-Based Semantic Representation Learning: Differentiable Biophysical Sequence Optimization

### ChemiSearch: End-to-End Mapping of Natural Language Semantics to DNA Hybridization Thermodynamics

## Section 1 — Cell 1: Environment, Imports, and Deterministic Seeding
Initialize deterministic seeding, CUDA/GPU verification, and workspace directories.


In [ ]:
# =============================================================================
# Cell 1: Environment, Imports, and Deterministic Seeding
# =============================================================================

import os
import gc
import copy
import random
import shutil
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.random_projection import GaussianRandomProjection
from torch.utils.data import TensorDataset, DataLoader
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 150

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED: int = 42

def seed_everything(seed: int) -> None:
    """Lock all random sources for guaranteed cross-run reproducibility."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # covers every GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# ---------------------------------------------------------------------------
# Workspace
# ---------------------------------------------------------------------------
WORKSPACE: str = "./dna_search_workspace"
for sub in ("data", "models", "figures", "export"):
    os.makedirs(os.path.join(WORKSPACE, sub), exist_ok=True)
os.makedirs("./figures", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"PyTorch       : {torch.__version__}")
print(f"Random seed   : {SEED}  (locked)")
print(f"Workspace     : {WORKSPACE}/")


## Section 2 — Cell 2: Figure 1 — System Architecture Diagram
Programmatically generates publication-quality Figure 1 illustrating the end-to-end differentiable antiparallel hybridization pipeline.

In [ ]:
# =============================================================================
# Cell 2: Figure 1 — System Architecture Diagram
# =============================================================================

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 7.5), dpi=300)
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)
ax.axis("off")

def box(ax, cx, cy, w, h, fc, ec="#333333", lw=1.5, alpha=1.0, pad=0.25):
    bl_x = cx - w/2
    bl_y = cy - h/2
    fancy = mpatches.FancyBboxPatch(
        (bl_x, bl_y), w, h,
        boxstyle=f"round,pad={pad}",
        facecolor=fc, edgecolor=ec, linewidth=lw, alpha=alpha, zorder=2
    )
    ax.add_patch(fancy)

def txt(ax, cx, cy, title, desc="", title_fs=11, desc_fs=9):
    if desc:
        ax.text(cx, cy + 0.15, title, fontsize=title_fs, fontweight="bold", ha="center", va="bottom", zorder=3)
        ax.text(cx, cy - 0.15, desc, fontsize=desc_fs, color="#444444", ha="center", va="top", zorder=3)
    else:
        ax.text(cx, cy, title, fontsize=title_fs, fontweight="bold", ha="center", va="center", zorder=3)

def arrow(ax, x1, y1, x2, y2, color="#333333", ls="-", lw=2.0):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color=color, lw=lw,
                        mutation_scale=20, linestyle=ls),
        zorder=1
    )

C_NLP  = "#e3f2fd"   # blue   — NLP / embedding space
C_ENC  = "#fff3e0"   # orange — encoder
C_DNA  = "#e8f5e9"   # green  — DNA / biology
C_PHY  = "#fce4ec"   # pink   — physics / thermodynamics
C_LOSS = "#f3e5f5"   # purple — loss

# Row 1 (y = 6.5)
box(ax, 2.2, 6.5, 2.0, 0.9, C_NLP)
txt(ax, 2.2, 6.5, "Input Text", '"A patient with\nhypertension..."')

box(ax, 7.0, 6.5, 3.4, 0.9, C_NLP)
txt(ax, 7.0, 6.5, "Sentence Transformer", "all-MiniLM-L6-v2\n(Frozen Teacher)")

box(ax, 11.8, 6.5, 2.0, 0.9, C_NLP)
txt(ax, 11.8, 6.5, "Embedding", "384-dim\nFloat32 vector")

# Row 2 (y = 4.0)
box(ax, 9.3, 4.0, 5.2, 1.1, C_ENC)
txt(ax, 9.3, 4.0, "ResidualMLPEncoder", 
    "384->512->512->512->(128x4)  |  GELU + LayerNorm + Dropout(0.1)\n"
    "Train: Straight-Through Gumbel-Softmax (tau: 1.0->0.2)\n"
    "Inference: Deterministic Argmax One-Hot (hard=True)")

box(ax, 14.2, 4.0, 2.0, 0.9, C_DNA)
txt(ax, 14.2, 4.0, "DNA Sequence", "128 bp\nA / C / G / T")

# Row 3 (y = 1.5)
box(ax, 13.0, 1.5, 3.8, 1.1, C_PHY)
txt(ax, 13.0, 1.5, "Thermodynamic Surrogate", 
    "SantaLucia NN  |  T=348.15K (75°C)\n"
    r"Antiparallel $\Delta G = \Delta H - T\Delta S$  |  Physical Mismatches" "\n"
    "Biological Constraints (Homopolymer >= 3 & GC balance)")

box(ax, 8.0, 1.5, 1.6, 0.9, C_PHY)
txt(ax, 8.0, 1.5, r"$-\Delta G$", "Affinity Score", title_fs=14)

box(ax, 4.0, 1.5, 2.6, 0.9, C_LOSS)
txt(ax, 4.0, 1.5, "Pearson Loss", "1 - r(affinity, target)\nTarget: similarity [0,1]")

# Draw Arrows
arrow(ax, 3.45, 6.5, 5.05, 6.5)       # Input -> Teacher
arrow(ax, 8.95, 6.5, 10.55, 6.5)      # Teacher -> Embed
arrow(ax, 11.8, 5.8, 11.8, 4.8)       # Embed -> Encoder
arrow(ax, 12.15, 4.0, 12.95, 4.0)     # Encoder -> DNA
arrow(ax, 14.2, 3.3, 14.2, 2.3)       # DNA -> Surrogate
arrow(ax, 10.85, 1.5, 9.05, 1.5)      # Surrogate -> DeltaG
arrow(ax, 6.95, 1.5, 5.55, 1.5)       # DeltaG -> Loss

# Target score path
ax.plot([2.2, 2.2, 2.7], [5.8, 1.5, 1.5], color="#555555", lw=2.0, ls=":", zorder=1)
arrow(ax, 2.5, 1.5, 2.7, 1.5, color="#555555", ls=":")
ax.text(2.35, 3.65, "Similarity Score (Target)", rotation=90, va="center", ha="left", 
        color="#555555", fontsize=10, fontweight="bold", zorder=3)

# Gradient path
ax.plot([4.0, 4.0, 6.4], [2.2, 3.3, 3.3], color="#d62728", lw=2.0, ls="--", zorder=1)
arrow(ax, 6.2, 3.3, 6.4, 3.3, color="#d62728", ls="--")
ax.text(4.2, 2.9, "Straight-Through gradient", color="#d62728", fontsize=10, 
        fontweight="bold", style="italic", zorder=3)

# Sequence 2 indicator
ax.annotate("", xy=(13.1, 2.3), xytext=(13.1, 4.0),
            arrowprops=dict(arrowstyle="->", color="#888888", lw=1.5, linestyle="--"), zorder=1)
ax.text(13.1, 4.15, "Sequence 2\n(antiparallel pairing)", color="#777777", fontsize=9, ha="center", va="bottom", zorder=3)

# Legend & Title
legend_items = [
    mpatches.Patch(facecolor=C_NLP,  edgecolor="#333", label="NLP / Embedding space"),
    mpatches.Patch(facecolor=C_ENC,  edgecolor="#333", label="Encoder (trainable, 987K params)"),
    mpatches.Patch(facecolor=C_DNA,  edgecolor="#333", label="DNA sequence space (128 bp)"),
    mpatches.Patch(facecolor=C_PHY,  edgecolor="#333", label="Biophysical surrogate (fixed physics)"),
    mpatches.Patch(facecolor=C_LOSS, edgecolor="#333", label="Differentiable loss"),
]
ax.legend(handles=legend_items, loc="upper right", bbox_to_anchor=(1.0, 1.05),
          fontsize=9, framealpha=1.0, edgecolor="#cccccc")

ax.set_title(
    "Figure 1: ChemiSearch System Architecture\n"
    "End-to-end differentiable text-to-DNA semantic encoding pipeline",
    fontweight="bold", fontsize=14, pad=20,
)

fig_path = os.path.join(WORKSPACE, "figures", "fig1_architecture.png")
fig.savefig(fig_path, bbox_inches="tight")
fig.savefig("./figures/fig1_architecture.png", bbox_inches="tight")
plt.show()
print(f"Figure 1 (architecture diagram) saved: {fig_path}")


## Section 3 — Cell 3: Data Acquisition & Multi-Domain Benchmark Loading
Downloads and standardizes training corpora (STS-B train, AllNLI triplets), in-domain held-out test split (STS-B test), and zero-shot out-of-domain benchmarks (BIOSSES biomedical, SICK-R commonsense, STS17 cross-lingual).


In [ ]:
# =============================================================================
# Cell 3: Data Acquisition & Multi-Domain Benchmark Loading
# =============================================================================

def load_datasets() -> dict:
    """
    Download and prepare semantic-similarity benchmarks.

    Training datasets
    -----------------
    - STS-B (train + validation) : human-annotated sentence pairs (scores 0-5)
    - AllNLI (30,000 triplets)   : anchor / positive / negative triples

    Held-out Evaluation datasets
    ----------------------------
    - STS-B test     : in-domain held-out test split
    - BIOSSES        : zero-shot biomedical sentence similarity (PubMed)
    - SICK-R         : zero-shot compositional / commonsense English
    - STS17 (en-en)  : zero-shot cross-lingual STS benchmark (English subset)

    All scores normalised to [0, 1] to unify training signal scale.
    """
    print("Loading datasets ...")

    # -- Training & In-Domain -------------------------------------------------
    stsb        = load_dataset("mteb/stsbenchmark-sts")
    train_df    = pd.DataFrame(stsb["train"])
    val_df      = pd.DataFrame(stsb["validation"])
    stsb_test   = pd.DataFrame(stsb["test"])

    nli_raw     = load_dataset("sentence-transformers/all-nli", "triplet", split="train")
    nli_df      = pd.DataFrame(nli_raw.shuffle(seed=SEED).select(range(30_000)))

    # -- Zero-shot test sets --------------------------------------------------
    biosses_df  = pd.DataFrame(load_dataset("mteb/biosses-sts", split="test"))
    sickr_df    = pd.DataFrame(load_dataset("mteb/sickr-sts", split="test"))
    sts17_raw   = load_dataset("mteb/sts17-crosslingual-sts", "en-en", split="test")
    sts17_df    = pd.DataFrame(sts17_raw)

    # -- Normalise all scores to [0, 1] ---------------------------------------
    for df, divisor in [
        (train_df,   5.0),
        (val_df,     5.0),
        (stsb_test,  5.0),
        (biosses_df, 5.0),
        (sickr_df,   5.0),
        (sts17_df,   5.0),
    ]:
        df["score"] = df["score"] / divisor

    print(f"  STS-B   train  : {len(train_df):>6}  pairs (in-domain training)")
    print(f"  STS-B   val    : {len(val_df):>6}  pairs (in-domain validation)")
    print(f"  AllNLI  train  : {len(nli_df):>6}  triplets (training)")
    print(f"  STS-B   test   : {len(stsb_test):>6}  pairs (in-domain held-out test)")
    print(f"  BIOSSES test   : {len(biosses_df):>6}  pairs (zero-shot, biomedical)")
    print(f"  SICK-R  test   : {len(sickr_df):>6}  pairs (zero-shot, commonsense)")
    print(f"  STS17   test   : {len(sts17_df):>6}  pairs (zero-shot, cross-lingual)")

    return dict(
        train_df=train_df, val_df=val_df, nli_df=nli_df,
        stsb_test=stsb_test, biosses_df=biosses_df,
        sickr_df=sickr_df, sts17_df=sts17_df,
    )

dsets = load_datasets()
train_df   = dsets["train_df"]
val_df     = dsets["val_df"]
nli_df     = dsets["nli_df"]
stsb_test  = dsets["stsb_test"]
biosses_df = dsets["biosses_df"]
sickr_df   = dsets["sickr_df"]
sts17_df   = dsets["sts17_df"]


## Section 4 — Cell 4: Frozen Teacher Embeddings & Persistent Caching
Generates 384-dimensional dense semantic vectors using all-MiniLM-L6-v2 and caches them locally for instant reuse.

In [ ]:
# =============================================================================
# Cell 4: Frozen Teacher Embeddings & Persistent Caching
# =============================================================================

def encode_and_cache() -> tuple:
    """
    Encode text with teacher model; save tensors to disk for instant reload.
    Returns train_ds, val_e1, val_e2, val_scores, test_data.
    """
    cache = os.path.join(WORKSPACE, "data", "embeddings.pt")

    if os.path.exists(cache):
        print("Cache found. Loading embeddings from disk ...")
        data = torch.load(cache, weights_only=False)
        return (
            data["train_ds"],
            data["val_e1"], data["val_e2"], data["val_scores"],
            data["test_data"],
        )

    print("Encoding from scratch (one-time cost) ...")
    teacher = SentenceTransformer("all-MiniLM-L6-v2")

    # -- STS-B in-domain training pairs --------------------------------------
    stsb_e1  = teacher.encode(train_df["sentence1"].tolist(), convert_to_tensor=True)
    stsb_e2  = teacher.encode(train_df["sentence2"].tolist(), convert_to_tensor=True)
    stsb_tgt = torch.tensor(train_df["score"].values, dtype=torch.float32)

    # -- AllNLI triplets -----------------------------------------------------
    nli_anc = teacher.encode(nli_df["anchor"].tolist(),   convert_to_tensor=True)
    nli_pos = teacher.encode(nli_df["positive"].tolist(), convert_to_tensor=True)
    nli_neg = teacher.encode(nli_df["negative"].tolist(), convert_to_tensor=True)
    pos_tgt = (torch.cosine_similarity(nli_anc, nli_pos) + 1.0) / 2.0
    neg_tgt = (torch.cosine_similarity(nli_anc, nli_neg) + 1.0) / 2.0

    # -- Combine training pairs ----------------------------------------------
    all_e1   = torch.cat([stsb_e1.cpu(),  nli_anc.cpu(), nli_anc.cpu()])
    all_e2   = torch.cat([stsb_e2.cpu(),  nli_pos.cpu(), nli_neg.cpu()])
    all_tgt  = torch.cat([stsb_tgt.cpu(), pos_tgt.cpu(), neg_tgt.cpu()])
    train_ds = TensorDataset(all_e1, all_e2, all_tgt)

    # -- In-Domain Validation ------------------------------------------------
    val_e1     = teacher.encode(val_df["sentence1"].tolist(), convert_to_tensor=True).cpu()
    val_e2     = teacher.encode(val_df["sentence2"].tolist(), convert_to_tensor=True).cpu()
    val_scores = val_df["score"].values

    # -- Evaluation sets: 1 in-domain held-out + 3 zero-shot -----------------
    test_data: Dict[str, Dict] = {}
    for name, df in [
        ("STS-B",   stsb_test),
        ("BIOSSES", biosses_df),
        ("SICK-R",  sickr_df),
        ("STS17",   sts17_df),
    ]:
        test_data[name] = {
            "e1":     teacher.encode(df["sentence1"].tolist(), convert_to_tensor=True).cpu(),
            "e2":     teacher.encode(df["sentence2"].tolist(), convert_to_tensor=True).cpu(),
            "scores": df["score"].values,
        }

    torch.save(dict(
        train_ds=train_ds, val_e1=val_e1, val_e2=val_e2,
        val_scores=val_scores, test_data=test_data,
    ), cache)
    print("Embeddings cached.")

    return train_ds, val_e1, val_e2, val_scores, test_data

train_ds, val_e1, val_e2, val_scores, test_data = encode_and_cache()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"Training pairs / samples : {len(train_ds):,}")
print(f"In-domain held-out test  : STS-B")
print(f"Zero-shot test sets      : BIOSSES, SICK-R, STS17")


## Section 5 — Cell 5: Neural Architecture & Biophysical Thermodynamic Surrogate
Defines PearsonCorrelationLoss, ResidualMLPEncoder with Straight-Through Gumbel-Softmax training and deterministic evaluation, and BulletproofThermodynamicSurrogate with antiparallel hybridization, SantaLucia 1998 nearest-neighbor physics, physical base-pair mismatch matrix, and biological viability constraints.


In [ ]:
# =============================================================================
# Cell 5: Neural Architecture & Biophysical Thermodynamic Surrogate
# =============================================================================

class PearsonCorrelationLoss(nn.Module):
    """
    Optimises rank alignment between predicted affinity and target scores.
    Loss = 1.0 - Pearson r.
    """
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        p = pred   - pred.mean()
        t = target - target.mean()
        cov = (p * t).mean()
        std_p = p.std(unbiased=False)
        std_t = t.std(unbiased=False)
        r = cov / (std_p * std_t + 1e-8)
        return 1.0 - r

# ---------------------------------------------------------------------------
# Encoder
# ---------------------------------------------------------------------------
class ResidualMLPEncoder(nn.Module):
    """
    Maps 384-dim sentence embedding to (seq_len, 4) discrete one-hot DNA tensor.
    Training uses Straight-Through Gumbel-Softmax relaxation.
    Evaluation uses deterministic argmax one-hot projection to eliminate stochastic noise.
    """
    def __init__(
        self,
        in_dim: int = 384,
        hidden: int = 512,
        seq_len: int = 128,
    ) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.proj    = nn.Linear(in_dim, hidden)
        self.block1  = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.1),
        )
        self.block2  = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.1),
        )
        self.to_dna  = nn.Linear(hidden, seq_len * 4)

    def forward(
        self,
        x: torch.Tensor,
        tau: float = 1.0,
        hard: bool = True,
    ) -> torch.Tensor:
        h = F.gelu(self.proj(x))
        h = h + self.block1(h)
        h = h + self.block2(h)
        logits = self.to_dna(h).view(-1, self.seq_len, 4)
        if self.training:
            return F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)
        else:
            # Deterministic categorical selection during evaluation (no stochastic Gumbel noise)
            idx = logits.argmax(dim=-1)
            return F.one_hot(idx, num_classes=4).float()

# ---------------------------------------------------------------------------
# Thermodynamic Surrogate
# ---------------------------------------------------------------------------
class BulletproofThermodynamicSurrogate(nn.Module):
    """
    Differentiable biophysical surrogate for DNA duplex hybridisation thermodynamics.

    Biophysical Foundations:
    1. Antiparallel Hybridisation: In nature, DNA duplexes hybridise in an
       antiparallel orientation (5'-to-3' pairs against 3'-to-5'). Strand 2 is
       reversed so that position k on strand 1 faces position (L-1-k) on strand 2.
    2. SantaLucia (1998) Unified Nearest-Neighbour (NN) Parameters:
       Calculates duplex enthalpy (dH) and entropy (dS) for matched dinucleotide steps:
       dG = dH - T * dS / 1000  (Operating T = 348.15 K / 75°C).
    3. Direct Physical Base-Pair Mismatch Matrix:
       Watson-Crick pairs (A-T, T-A, C-G, G-C) receive 0 penalty.
       G-T / T-G wobble pairs receive a mild penalty (0.5 kcal/mol).
       Purine-purine clashes (A-A, G-G, A-G, G-A) receive 2.0 kcal/mol.
       Pyrimidine / other clashes receive 1.5 kcal/mol.
    4. Biological Viability Constraints:
       - Homopolymer penalty: penalises runs of >= 3 identical consecutive bases.
       - GC-content regulation: penalises sequences straying from optimal 40-60% GC.
    """

    # SantaLucia 1998 nearest-neighbour parameters (rows/cols: A C G T)
    _DH = [
        [-7.9,  -8.4,  -7.8,  -7.2],
        [-8.5,  -8.0, -10.6,  -7.8],
        [-8.2,  -9.8,  -8.0,  -8.4],
        [-7.2,  -8.2,  -8.5,  -7.9],
    ]
    _DS = [
        [-22.2, -22.4, -21.0, -20.4],
        [-22.7, -19.9, -27.2, -21.0],
        [-22.2, -24.4, -19.9, -22.4],
        [-21.3, -22.2, -22.7, -22.2],
    ]
    _MM = [
        #   A    C    G    T   (dna2 antiparallel)
        [ 2.0, 1.5, 2.0, 0.0 ],  # A (dna1)
        [ 1.5, 1.5, 0.0, 1.5 ],  # C
        [ 2.0, 0.0, 2.0, 0.5 ],  # G
        [ 0.0, 1.5, 0.5, 1.5 ],  # T
    ]

    def __init__(self, seq_len: int = 128, temperature: float = 348.15) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.T       = temperature
        self.register_buffer("dH", torch.tensor(self._DH, dtype=torch.float32))
        self.register_buffer("dS", torch.tensor(self._DS, dtype=torch.float32))
        self.register_buffer("MM", torch.tensor(self._MM, dtype=torch.float32))

    @staticmethod
    def _wc_complement(dna: torch.Tensor) -> torch.Tensor:
        """Watson-Crick complement: A<->T (0<->3), C<->G (1<->2)."""
        return dna[:, :, [3, 2, 1, 0]]

    def _biological_constraints(self, dna1: torch.Tensor, dna2: torch.Tensor) -> torch.Tensor:
        """
        Differentiable biological viability constraints:
        1. Homopolymer run penalty: penalise 3+ identical consecutive bases.
        2. GC content balance: penalise sequences deviating from target 50% GC.
        """
        hp1 = (dna1[:, :-2] * dna1[:, 1:-1] * dna1[:, 2:]).sum(dim=(-1, -2))
        hp2 = (dna2[:, :-2] * dna2[:, 1:-1] * dna2[:, 2:]).sum(dim=(-1, -2))
        homopolymer_penalty = 0.5 * (hp1 + hp2)

        gc1 = (dna1[:, :, 1] + dna1[:, :, 2]).sum(dim=1) / self.seq_len
        gc2 = (dna2[:, :, 1] + dna2[:, :, 2]).sum(dim=1) / self.seq_len
        gc_penalty = ((gc1 - 0.5)**2 + (gc2 - 0.5)**2) * 2.0

        return homopolymer_penalty + gc_penalty

    def forward(self, dna1: torch.Tensor, dna2: torch.Tensor) -> torch.Tensor:
        """
        Compute predicted hybridisation affinity (-DeltaG) for each pair.
        dna1, dna2: (B, seq_len, 4)
        """
        # Antiparallel duplex alignment (reverse strand 2 along length)
        dna2_rev = torch.flip(dna2, dims=[1])
        dna2_rc  = self._wc_complement(dna2_rev)

        # Watson-Crick match indicator per duplex position: (B, L)
        match = (dna1 * dna2_rc).sum(dim=-1)

        # Nearest-neighbour free energy along strand 1
        dG = self.dH - self.T * self.dS / 1000.0   # (4, 4)
        step = torch.einsum("bni,ij,bnj->bn", dna1[:, :-1], dG, dna1[:, 1:])

        # Both consecutive positions must match for a stable NN duplex step
        pair_mask = match[:, :-1] * match[:, 1:]   # (B, L-1)
        nn_energy = (step * pair_mask).sum(dim=1)  # (B,)

        # Mismatch penalty via direct physical base-pair lookup
        mismatch = torch.einsum("bni,ij,bnj->b", dna1, self.MM, dna2_rev)

        # Biological viability constraints (homopolymers + GC balance)
        bio_pen = self._biological_constraints(dna1, dna2)

        total_energy = nn_energy + mismatch + bio_pen

        # Return affinity (-DeltaG): more negative energy -> higher binding affinity
        return -total_energy

_enc_params = sum(p.numel() for p in ResidualMLPEncoder().parameters())
print(f"ResidualMLPEncoder          : {_enc_params:,} trainable parameters")
print(f"ThermodynamicSurrogate      : 0 learnable parameters (fixed physics)")
print("Architecture defined successfully.")


## Section 6 — Cell 6: End-to-End Training Pipeline with Straight-Through Gumbel-Softmax
Optimizes encoder parameters via AdamW and Cosine Annealing. Straight-Through Gumbel-Softmax (hard=True) ensures that the network trains directly on discrete nucleotide representations, resolving train-evaluation distribution collapse.

In [ ]:
# =============================================================================
# Cell 6: End-to-End Training Pipeline with Resumable Checkpointing
# =============================================================================

encoder   = ResidualMLPEncoder().to(device)
predictor = BulletproofThermodynamicSurrogate().to(device)
criterion = PearsonCorrelationLoss()

optimizer = torch.optim.AdamW(encoder.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

loader     = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)
ckpt_path  = os.path.join(WORKSPACE, "models", "checkpoint.pth")

EPOCHS   = 25
PATIENCE = 5

if os.path.exists(ckpt_path):
    print("Checkpoint found. Resuming from saved state ...")
    ck        = torch.load(ckpt_path, weights_only=False)
    encoder.load_state_dict(ck["encoder"])
    best_rho  = ck["best_rho"]
    start_ep  = ck.get("epoch", 0) + 1
    pat_ctr   = ck.get("pat_ctr", 0)
    for _ in range(ck.get("epoch", 0)):
        scheduler.step()
    print(f"  Resumed at epoch {start_ep}  |  best val rho so far = {best_rho:.4f}")
else:
    print("No checkpoint found. Starting training from scratch ...")
    best_rho = -1.0
    start_ep = 1
    pat_ctr  = 0

print(f"{'Epoch':>5}  {'Train Loss':>11}  {'Val Rho':>8}  {'Status'}")
print("-" * 48)

for epoch in range(start_ep, EPOCHS + 1):
    encoder.train()
    tau = max(0.2, 1.0 - epoch * 0.04)
    epoch_loss = 0.0

    for b_e1, b_e2, b_tgt in loader:
        optimizer.zero_grad()
        d1  = encoder(b_e1.to(device), tau=tau, hard=True)
        d2  = encoder(b_e2.to(device), tau=tau, hard=True)
        aff = predictor(d1, d2)
        loss = criterion(aff, b_tgt.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()

    # -- Validation (Deterministic argmax, no Gumbel noise) ------------------
    encoder.eval()
    with torch.no_grad():
        v_d1  = encoder(val_e1.to(device))
        v_d2  = encoder(val_e2.to(device))
        v_aff = predictor(v_d1, v_d2).cpu().numpy()

    val_rho, _ = stats.spearmanr(val_scores, v_aff)

    if val_rho > best_rho:
        best_rho, pat_ctr = val_rho, 0
        torch.save({
            "encoder":  encoder.state_dict(),
            "best_rho": best_rho,
            "epoch":    epoch,
            "pat_ctr":  pat_ctr,
        }, ckpt_path)
        status = "SAVED"
    else:
        pat_ctr += 1
        if os.path.exists(ckpt_path):
            ck = torch.load(ckpt_path, weights_only=False)
            ck["pat_ctr"] = pat_ctr
            ck["epoch"]   = epoch
            torch.save(ck, ckpt_path)
        status = f"no improvement ({pat_ctr}/{PATIENCE})"

    print(f"{epoch:>5}  {epoch_loss/len(loader):>11.4f}  {val_rho:>8.4f}  {status}")

    if pat_ctr >= PATIENCE:
        print("Early stopping triggered.")
        break

# -- Load best weights -------------------------------------------------------
if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, weights_only=False)
    encoder.load_state_dict(ck["encoder"])
    print(f"Best model loaded  (val Spearman rho = {ck['best_rho']:.4f})")


## Section 7 — Cell 7: Figure 3 — Multi-Domain Evaluation (In-Domain & Zero-Shot)
Evaluates the trained model on STS-B (in-domain held-out) and zero-shot transfer on BIOSSES (biomedical), SICK-R (commonsense), and STS17 (cross-lingual); saves Figure 3.


In [ ]:
# =============================================================================
# Cell 7: Figure 3 — Multi-Domain Evaluation (In-Domain & Zero-Shot)
# =============================================================================

plt.style.use("seaborn-v0_8-whitegrid")
encoder.eval()

EVAL_DATASETS = [
    ("STS-B",   "#2ca02c", "In-Domain (Held-Out)"),
    ("BIOSSES", "#d62728", "Zero-Shot (Biomedical)"),
    ("SICK-R",  "#9467bd", "Zero-Shot (Commonsense)"),
    ("STS17",   "#ff7f0e", "Zero-Shot (Cross-Lingual)"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 11), dpi=300)
axes = axes.flatten()

print("Multi-Domain Evaluation (4 benchmarks)")
print(f"  {'Dataset':<12}  {'Domain / Split Type':<26}  {'n':>5}  {'Spearman rho':>13}  {'p-value':>12}")
print("  " + "-" * 76)

for ax, (name, color, domain) in zip(axes, EVAL_DATASETS):
    td = test_data[name]
    with torch.no_grad():
        d1  = encoder(td["e1"].to(device))
        d2  = encoder(td["e2"].to(device))
        aff = predictor(d1, d2).cpu().numpy()

    sc      = td["scores"]
    rho, pv = stats.spearmanr(sc, aff)
    pv_str  = f"{pv:.3e}"

    print(f"  {name:<12}  {domain:<26}  {len(sc):>5}  {rho:>13.4f}  {pv_str:>12}")

    ax.scatter(aff, sc, alpha=0.40, s=16, c=color, edgecolors="k", linewidths=0.2)
    x_line = np.linspace(aff.min(), aff.max(), 200)
    m_c, b_c = np.polyfit(aff, sc, 1)
    ax.plot(x_line, m_c * x_line + b_c, "k--", lw=2,
            label=f"rho = {rho:.3f}\np  = {pv_str}")
    ax.set_title(f"{name}: {domain} (n={len(sc)})", fontweight="bold")
    ax.set_xlabel("Predicted Hybridisation Affinity (-DeltaG)")
    ax.set_ylabel("Human Semantic Score [0-1]")
    ax.legend(fontsize=9)

plt.suptitle(
    "Figure 3: Semantic Alignment Across Multi-Domain Benchmarks\n"
    "ChemiSearch evaluated on STS-B (held-out in-domain) and 3 zero-shot benchmarks",
    fontweight="bold", fontsize=13, y=1.01,
)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig3_zero_shot.png"), bbox_inches="tight")
fig.savefig("./figures/fig3_zero_shot.png", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")


## Section 8 — Cell 8: Figure 2 — Baseline Comparisons (Digital vs Biophysical)
Benchmarks ChemiSearch against Float32 Teacher, Int8, Binary Quantization, 256-bit LSH Hashing, and Random DNA lower bound; saves Figure 2.

In [ ]:
# =============================================================================
# Cell 8: Figure 2 — Baseline Comparisons (Digital vs Biophysical)
# =============================================================================

td     = test_data["STS-B"]
scores = td["scores"]
e1_np  = td["e1"].numpy()
e2_np  = td["e2"].numpy()

results: Dict[str, float] = {}

# 1. Float32 teacher (upper bound)
cos = torch.cosine_similarity(td["e1"], td["e2"]).numpy()
results["Float32 Teacher (UB)"] = stats.spearmanr(scores, cos)[0]

# 2. Int8 scalar quantisation
def _int8(x: np.ndarray) -> np.ndarray:
    lo, hi = x.min(), x.max()
    s = (hi - lo) / 255.0 + 1e-9
    return np.round((x - lo) / s) * s + lo

i8e1, i8e2 = _int8(e1_np), _int8(e2_np)
i8sim = np.einsum("bi,bi->b", i8e1, i8e2) / (
    np.linalg.norm(i8e1, axis=1) * np.linalg.norm(i8e2, axis=1) + 1e-9
)
results["Int8 Quantisation"] = stats.spearmanr(scores, i8sim)[0]

# 3. Binary quantisation
b1 = (e1_np > 0).astype(np.float32)
b2 = (e2_np > 0).astype(np.float32)
results["Binary Quantisation"] = stats.spearmanr(
    scores, 1.0 - np.mean(b1 != b2, axis=1)
)[0]

# 4. LSH (256-bit Gaussian random projections)
rp = GaussianRandomProjection(n_components=256, random_state=42)
rp.fit(np.vstack([e1_np, e2_np]))
l1 = (rp.transform(e1_np) > 0).astype(np.float32)
l2 = (rp.transform(e2_np) > 0).astype(np.float32)
results["LSH Hashing (256-bit)"] = stats.spearmanr(
    scores, 1.0 - np.mean(l1 != l2, axis=1)
)[0]

# 5. Random DNA (lower bound)
encoder.eval()
with torch.no_grad():
    r1 = F.one_hot(torch.randint(0, 4, (td["e1"].size(0), 128), device=device), 4).float()
    r2 = F.one_hot(torch.randint(0, 4, (td["e2"].size(0), 128), device=device), 4).float()
    results["Random DNA (LB)"] = stats.spearmanr(
        scores, predictor(r1, r2).cpu().numpy()
    )[0]

# 6. ChemiSearch (ours) - Deterministic evaluation
with torch.no_grad():
    d1 = encoder(td["e1"].to(device))
    d2 = encoder(td["e2"].to(device))
    results["ChemiSearch (Ours)"] = stats.spearmanr(
        scores, predictor(d1, d2).cpu().numpy()
    )[0]

# -- Print table -------------------------------------------------------------
ORDER = [
    "Random DNA (LB)", "Binary Quantisation", "LSH Hashing (256-bit)",
    "Int8 Quantisation", "Float32 Teacher (UB)", "ChemiSearch (Ours)",
]
print(f"  {'Method':<28}  Spearman rho")
print("  " + "-" * 42)
for k in ORDER:
    print(f"  {k:<28}  {results[k]:+.4f}")

# -- Bar chart ---------------------------------------------------------------
rhos   = [results[k] for k in ORDER]
colors = ["#7f7f7f", "#1f77b4", "#1f77b4", "#1f77b4", "#2ca02c", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
ax.barh(ORDER, rhos, color=colors, edgecolor="black", alpha=0.88)
ax.set_xlabel("STS-B Spearman rho (In-Domain Test)", fontweight="bold", fontsize=12)
ax.set_title(
    "Figure 2: Information Preserved — Biophysical DNA vs Digital Methods",
    fontweight="bold", fontsize=13,
)
ax.grid(axis="x", linestyle="--", alpha=0.6)
for i, v in enumerate(rhos):
    ax.text(max(v, 0.0) + 0.01, i, f"{v:+.3f}", va="center",
            fontweight="bold", fontsize=11)

ax.set_xlim(-0.1, 1.05)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig2_baselines.png"),
            bbox_inches="tight")
fig.savefig("./figures/fig2_baselines.png", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")


## Section 9 — Cell 9: Figure 4 — Biological Validity & Mutational Robustness
Evaluates GC-content distribution and bootstraps (200x) synthetic synthesis/sequencing error rates up to 20%; saves Figure 4.

In [ ]:
# =============================================================================
# Cell 9: Figure 4 — Biological Validity & Mutational Robustness
# =============================================================================

encoder.eval()
with torch.no_grad():
    d1_full = encoder(test_data["STS-B"]["e1"].to(device))
    d2_full = encoder(test_data["STS-B"]["e2"].to(device))

sc_full = test_data["STS-B"]["scores"]
N       = len(sc_full)

# -- GC content --------------------------------------------------------------
gc_pct = (d1_full[:, :, 1] + d1_full[:, :, 2]).sum(1).cpu().numpy() / 128.0 * 100.0

# -- Robustness with bootstrap CI (full test set) ----------------------------
ERROR_RATES = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]
BOOT_N      = 200
rho_mean, rho_lo, rho_hi = [], [], []

for er in ERROR_RATES:
    boot_rhos = []
    for _ in range(BOOT_N):
        idx  = np.random.choice(N, N, replace=True)
        nm1  = torch.rand(N, 128, device=device) < er
        nm2  = torch.rand(N, 128, device=device) < er
        rb1  = F.one_hot(torch.randint(0, 4, (N, 128), device=device), 4).float()
        rb2  = F.one_hot(torch.randint(0, 4, (N, 128), device=device), 4).float()
        md1  = torch.where(nm1.unsqueeze(-1), rb1, d1_full)
        md2  = torch.where(nm2.unsqueeze(-1), rb2, d2_full)
        with torch.no_grad():
            aff = predictor(md1, md2).cpu().numpy()
        r, _ = stats.spearmanr(sc_full[idx], aff[idx])
        boot_rhos.append(r)

    rho_mean.append(float(np.mean(boot_rhos)))
    rho_lo.append(float(np.percentile(boot_rhos, 2.5)))
    rho_hi.append(float(np.percentile(boot_rhos, 97.5)))

# -- Figure ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# Panel A
axes[0].hist(gc_pct, bins=25, color="#9467bd", edgecolor="k", alpha=0.75, density=True)
axes[0].axvline(50, color="r", ls="--", lw=1.5, label="Target 50%")
axes[0].axvspan(40, 60, color="green", alpha=0.1, label="Optimal range (40-60%)")
axes[0].set_title("A. GC Content Distribution of Generated DNA", fontweight="bold")
axes[0].set_xlabel("GC Content (%)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Panel B
ep = [e * 100 for e in ERROR_RATES]
axes[1].plot(ep, rho_mean, "o-", lw=2, color="#e377c2", label="Mean rho")
axes[1].fill_between(ep, rho_lo, rho_hi, alpha=0.25, color="#e377c2",
                     label="95% Bootstrap CI")
axes[1].set_title("B. Robustness: Synthesis / Sequencing Error (bootstrapped)",
                  fontweight="bold")
axes[1].set_xlabel("Mutation Rate (%)")
axes[1].set_ylabel("Semantic Preservation (Spearman rho)")
axes[1].legend()
axes[1].grid(True, ls="--", alpha=0.5)

plt.suptitle("Figure 4: Biological Validity and Error Robustness",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig4_bio_validity.png"),
            bbox_inches="tight")
fig.savefig("./figures/fig4_bio_validity.png", bbox_inches="tight")
plt.show()
print("Figure 4 saved.")
print(f"GC content — mean: {gc_pct.mean():.1f}%  "
      f"std: {gc_pct.std():.1f}%  "
      f"fraction in [40,60]%: {((gc_pct>=40)&(gc_pct<=60)).mean():.2f}")


## Section 10 — Cell 10: Figure 5 — Systematic Biophysical Ablation Study
Ablates individual components (mismatch penalty, biological viability constraints [homopolymer runs & GC balance], residual architecture) under controlled conditions; saves Figure 5.


In [ ]:
# =============================================================================
# Cell 10: Figure 5 — Systematic Biophysical Ablation Study
# =============================================================================

ABLATION_EPOCHS  = 8
ABLATION_PATIENCE = 3

def ablation_train_eval(enc_model: nn.Module, surr_model: nn.Module) -> float:
    """
    Train enc_model for a fixed schedule and return deterministic STS-B test rho.
    """
    enc  = enc_model.to(device)
    surr = surr_model.to(device)
    opt  = torch.optim.AdamW(enc.parameters(), lr=5e-4, weight_decay=1e-3)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    crit = PearsonCorrelationLoss()
    ldr  = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)

    best_val, pat, best_state = -1.0, 0, None
    for ep in range(1, ABLATION_EPOCHS + 1):
        enc.train()
        tau = max(0.2, 1.0 - ep * 0.05)
        for b1, b2, bt in ldr:
            opt.zero_grad()
            a = surr(enc(b1.to(device), tau=tau, hard=True), enc(b2.to(device), tau=tau, hard=True))
            crit(a, bt.to(device)).backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            opt.step()
        sch.step()

        enc.eval()
        with torch.no_grad():
            va = surr(enc(val_e1.to(device)),
                      enc(val_e2.to(device))).cpu().numpy()
        vr, _ = stats.spearmanr(val_scores, va)
        if vr > best_val:
            best_val, pat = vr, 0
            best_state = copy.deepcopy(enc.state_dict())
        else:
            pat += 1
        if pat >= ABLATION_PATIENCE:
            break

    if best_state is not None:
        enc.load_state_dict(best_state)
    enc.eval()
    with torch.no_grad():
        ta = surr(enc(test_data["STS-B"]["e1"].to(device)),
                  enc(test_data["STS-B"]["e2"].to(device))).cpu().numpy()
    rho, _ = stats.spearmanr(test_data["STS-B"]["scores"], ta)
    return rho

# -- Surrogate variants -------------------------------------------------------
class NoMismatchSurrogate(BulletproofThermodynamicSurrogate):
    """Removes the mismatch penalty term from the biophysical surrogate."""
    def forward(self, dna1, dna2):
        dna2_rev = torch.flip(dna2, dims=[1])
        dna2_rc  = self._wc_complement(dna2_rev)
        match = (dna1 * dna2_rc).sum(dim=-1)
        dG = self.dH - self.T * self.dS / 1000.0
        step = torch.einsum("bni,ij,bnj->bn", dna1[:, :-1], dG, dna1[:, 1:])
        nn_energy = (step * (match[:, :-1] * match[:, 1:])).sum(dim=1)
        bio_pen = self._biological_constraints(dna1, dna2)
        return -(nn_energy + bio_pen)

class NoBioConstraintSurrogate(BulletproofThermodynamicSurrogate):
    """Removes biological viability constraints (homopolymer and GC penalty)."""
    def forward(self, dna1, dna2):
        dna2_rev = torch.flip(dna2, dims=[1])
        dna2_rc  = self._wc_complement(dna2_rev)
        match = (dna1 * dna2_rc).sum(dim=-1)
        dG = self.dH - self.T * self.dS / 1000.0
        step = torch.einsum("bni,ij,bnj->bn", dna1[:, :-1], dG, dna1[:, 1:])
        nn_energy = (step * (match[:, :-1] * match[:, 1:])).sum(dim=1)
        mismatch = torch.einsum("bni,ij,bnj->b", dna1, self.MM, dna2_rev)
        return -(nn_energy + mismatch)

class LinearEncoder(nn.Module):
    """Shallow two-layer encoder without residual connections or normalisation."""
    def __init__(self, in_dim: int = 384, hidden: int = 512, seq_len: int = 128):
        super().__init__()
        self.seq_len = seq_len
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, seq_len * 4)

    def forward(self, x, tau=1.0, hard=True):
        h = F.relu(self.fc1(x))
        logits = self.fc2(h).view(-1, self.seq_len, 4)
        if self.training:
            return F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)
        else:
            return F.one_hot(logits.argmax(dim=-1), num_classes=4).float()

# -- Run ablations -----------------------------------------------------------
print("Running ablation study ...")
print(f"  {'Variant':<38}  {'Test Rho':>8}")
print("  " + "-" * 50)

ablation_results = {}

# Full model (deterministic)
encoder.eval()
with torch.no_grad():
    fa = predictor(encoder(test_data["STS-B"]["e1"].to(device)),
                   encoder(test_data["STS-B"]["e2"].to(device))).cpu().numpy()
ablation_results["Full Model (ChemiSearch)"] = stats.spearmanr(test_data["STS-B"]["scores"], fa)[0]
print(f"  {'Full Model (ChemiSearch)':<38}  {ablation_results['Full Model (ChemiSearch)']:>8.4f}  (pre-trained)")

for tag, enc_cls, surr_cls in [
    ("No Bio Constraints (Homopolymer/GC)", ResidualMLPEncoder, NoBioConstraintSurrogate),
    ("No Mismatch Penalty",                ResidualMLPEncoder, NoMismatchSurrogate),
    ("No Residual Blocks",                 LinearEncoder,      BulletproofThermodynamicSurrogate),
]:
    rho = ablation_train_eval(enc_cls(), surr_cls())
    ablation_results[tag] = rho
    print(f"  {tag:<38}  {rho:>8.4f}")

ablation_results["Random DNA (LB)"] = results["Random DNA (LB)"]
print(f"  {'Random DNA (LB)':<38}  {ablation_results['Random DNA (LB)']:>8.4f}  (pre-computed)")

# -- Figure 5 ----------------------------------------------------------------
abl_order = [
    "Random DNA (LB)",
    "No Residual Blocks",
    "Full Model (ChemiSearch)",
    "No Bio Constraints (Homopolymer/GC)",
    "No Mismatch Penalty",
]
abl_rhos   = [ablation_results[k] for k in abl_order]
abl_colors = ["#7f7f7f", "#aec7e8", "#d62728", "#ffbb78", "#98df8a"]

fig, ax = plt.subplots(figsize=(10, 5.2), dpi=300)
ax.barh(abl_order, abl_rhos, color=abl_colors, edgecolor="black", alpha=0.88)
ax.set_xlabel("STS-B Spearman rho (In-Domain Held-Out Test)", fontweight="bold", fontsize=12)
ax.set_title("Figure 5: Biophysical Ablation Study — Component Contributions",
             fontweight="bold", fontsize=13)
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.axvline(0.0, color="black", ls="-", lw=1.0)
ax.axvline(ablation_results["Random DNA (LB)"], color="gray",
           ls=":", lw=1.5, label="Random DNA baseline")

for i, v in enumerate(abl_rhos):
    offset = 0.005 if v >= 0 else -0.035
    ax.text(v + offset, i, f"{v:+.4f}", va="center",
            fontweight="bold", fontsize=11)

ax.set_xlim(-0.06, max(max(abl_rhos) + 0.08, 0.28))
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig5_ablation.png"),
            bbox_inches="tight")
fig.savefig("./figures/fig5_ablation.png", bbox_inches="tight")
plt.show()
print("Figure 5 saved.")


## Section 11 — Cell 11: Figure 6 — Comprehensive Multi-Domain Visual Comparison
Renders unified publication-ready comparison tables and summary charts across digital baselines and multi-domain benchmarks; saves Figure 6.


In [ ]:
# =============================================================================
# Cell 11: Figure 6 — Comprehensive Visual Comparison Tables
# =============================================================================

import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

encoder.eval()

# Synchronized deterministic evaluation
td_stsb    = test_data["STS-B"]
sc_stsb    = td_stsb["scores"]
e1_np      = td_stsb["e1"].numpy()
e2_np      = td_stsb["e2"].numpy()

with torch.no_grad():
    d1 = encoder(td_stsb["e1"].to(device))
    d2 = encoder(td_stsb["e2"].to(device))
    chemisearch_stsb_rho = stats.spearmanr(sc_stsb, predictor(d1, d2).cpu().numpy())[0]

baseline_rhos = {}
cos = torch.cosine_similarity(td_stsb["e1"], td_stsb["e2"]).numpy()
baseline_rhos["Float32 Teacher (UB)"] = stats.spearmanr(sc_stsb, cos)[0]

def _int8(x: np.ndarray) -> np.ndarray:
    lo, hi = x.min(), x.max()
    return np.round((x - lo) / ((hi - lo) / 255.0 + 1e-9)) * ((hi - lo) / 255.0 + 1e-9) + lo

i8e1, i8e2 = _int8(e1_np), _int8(e2_np)
i8sim = np.einsum("bi,bi->b", i8e1, i8e2) / (
    np.linalg.norm(i8e1, axis=1) * np.linalg.norm(i8e2, axis=1) + 1e-9)
baseline_rhos["Int8 Quantisation"]      = stats.spearmanr(sc_stsb, i8sim)[0]

b1 = (e1_np > 0).astype(np.float32)
b2 = (e2_np > 0).astype(np.float32)
baseline_rhos["Binary Quantisation"]   = stats.spearmanr(sc_stsb, 1.0 - np.mean(b1 != b2, axis=1))[0]

rp = GaussianRandomProjection(n_components=256, random_state=42)
rp.fit(np.vstack([e1_np, e2_np]))
l1 = (rp.transform(e1_np) > 0).astype(np.float32)
l2 = (rp.transform(e2_np) > 0).astype(np.float32)
baseline_rhos["LSH Hashing (256-bit)"] = stats.spearmanr(sc_stsb, 1.0 - np.mean(l1 != l2, axis=1))[0]

with torch.no_grad():
    r1 = F.one_hot(torch.randint(0, 4, (td_stsb["e1"].size(0), 128), device=device), 4).float()
    r2 = F.one_hot(torch.randint(0, 4, (td_stsb["e2"].size(0), 128), device=device), 4).float()
    baseline_rhos["Random DNA (LB)"]   = stats.spearmanr(sc_stsb, predictor(r1, r2).cpu().numpy())[0]

baseline_rhos["ChemiSearch (Ours)"]    = chemisearch_stsb_rho

# -- Multi-domain evaluation -------------------------------------------------
multidomain = {}
for name, tag in [
    ("STS-B",   "In-Domain (Held-Out)"),
    ("SICK-R",  "Zero-Shot (Commonsense)"),
    ("BIOSSES", "Zero-Shot (Biomedical)"),
    ("STS17",   "Zero-Shot (Cross-Lingual)"),
]:
    tdd = test_data[name]
    with torch.no_grad():
        ta = predictor(encoder(tdd["e1"].to(device)),
                       encoder(tdd["e2"].to(device))).cpu().numpy()
    rho, pval = stats.spearmanr(tdd["scores"], ta)
    multidomain[name] = {"domain": tag, "n": len(tdd["scores"]), "rho": rho, "pval": pval}

# ---------------------------------------------------------------------------
# Figure 6 layout
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(19, 14), dpi=300)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax_tbl_base = fig.add_subplot(gs[0, 0])
ax_tbl_zero = fig.add_subplot(gs[0, 1])
ax_bar_base = fig.add_subplot(gs[1, 0])
ax_bar_zero = fig.add_subplot(gs[1, 1])

# Panel A — Baseline table
ax_tbl_base.axis("off")
base_order = ["Float32 Teacher (UB)", "Int8 Quantisation", "Binary Quantisation", "LSH Hashing (256-bit)", "ChemiSearch (Ours)", "Random DNA (LB)"]
sorted_base = sorted(base_order, key=lambda k: baseline_rhos[k], reverse=True)

tbl_data, row_colors = [], []
for i, method in enumerate(sorted_base):
    rho = baseline_rhos[method]
    tbl_data.append([f"{i+1}", method, f"{rho:.4f}"])
    if "Ours" in method: row_colors.append(["#ffd6d6"] * 3)
    elif "UB" in method: row_colors.append(["#d6f5d6"] * 3)
    elif "LB" in method: row_colors.append(["#e8e8e8"] * 3)
    else:                row_colors.append(["#ddeeff"] * 3)

tbl_A = ax_tbl_base.table(
    cellText=tbl_data, colLabels=["Rank", "Method", "Spearman rho"],
    colWidths=[0.15, 0.55, 0.30],
    cellLoc="center", loc="center", cellColours=row_colors,
)
tbl_A.auto_set_font_size(False)
tbl_A.set_fontsize(11)
tbl_A.scale(1.0, 2.1)
for j in range(3):
    tbl_A[(0, j)].set_facecolor("#2c3e50")
    tbl_A[(0, j)].set_text_props(color="white", fontweight="bold")
ax_tbl_base.set_title("A.  Baseline Comparison — STS-B In-Domain Test (n=1,379)", fontweight="bold", fontsize=12, pad=12)

# Panel B — Multi-Domain table
ax_tbl_zero.axis("off")
zs_data = []
for name, v in multidomain.items():
    zs_data.append([name, v["domain"], f"{v['n']:,}", f"{v['rho']:+.4f}", f"{v['pval']:.2e}"])

tbl_B = ax_tbl_zero.table(
    cellText=zs_data, colLabels=["Dataset", "Domain / Split Type", "n", "Spearman rho", "p-value"],
    colWidths=[0.18, 0.38, 0.14, 0.18, 0.16],
    cellLoc="center", loc="center", cellColours=[["#ffd6d6"] * 5 for _ in range(len(zs_data))],
)
tbl_B.auto_set_font_size(False)
tbl_B.set_fontsize(10.5)
tbl_B.scale(1.0, 2.5)
for j in range(5):
    tbl_B[(0, j)].set_facecolor("#2c3e50")
    tbl_B[(0, j)].set_text_props(color="white", fontweight="bold")
ax_tbl_zero.text(0.5, 0.05, "Note: STS-B is in-domain held-out test; SICK-R, BIOSSES, and STS17 represent zero-shot transfer.",
                 transform=ax_tbl_zero.transAxes, ha="center", fontsize=9.5, style="italic", color="#555555")
ax_tbl_zero.set_title("B.  Multi-Domain Generalisation — ChemiSearch", fontweight="bold", fontsize=12, pad=12)

# Panel C — Baseline bar chart
bar_order  = ["Random DNA (LB)", "ChemiSearch (Ours)", "Binary Quantisation", "LSH Hashing (256-bit)", "Int8 Quantisation", "Float32 Teacher (UB)"]
bar_rhos   = [baseline_rhos[k] for k in bar_order]
bar_colors = ["#d62728" if "Ours" in k else "#2ca02c" if "UB" in k else "#7f7f7f" if "LB" in k else "#1f77b4" for k in bar_order]
ax_bar_base.barh(bar_order, bar_rhos, color=bar_colors, edgecolor="black", alpha=0.88)
ax_bar_base.set_xlabel("STS-B Spearman rho", fontweight="bold")
ax_bar_base.set_title("C.  Semantic Preservation — All Methods", fontweight="bold", fontsize=12)
ax_bar_base.grid(axis="x", linestyle="--", alpha=0.5)
for i, v in enumerate(bar_rhos):
    ax_bar_base.text(max(v, 0) + 0.01, i, f"{v:+.3f}", va="center", fontweight="bold", fontsize=10)
ax_bar_base.set_xlim(-0.08, 1.0)

# Panel D — Domain bar chart
zs_names = list(multidomain.keys())
zs_rhos  = [multidomain[n]["rho"] for n in zs_names]
zs_cols  = ["#2ca02c" if "In-Domain" in multidomain[n]["domain"] else "#1f77b4" if multidomain[n]["rho"] > 0 else "#d62728" for n in zs_names]
ax_bar_zero.bar(zs_names, zs_rhos, color=zs_cols, edgecolor="black", alpha=0.88, width=0.45)
ax_bar_zero.axhline(0, color="black", lw=1)
ax_bar_zero.set_ylabel("Spearman rho", fontweight="bold")
ax_bar_zero.set_title("D.  Performance Across Multi-Domain Benchmarks", fontweight="bold", fontsize=12)
ax_bar_zero.grid(axis="y", linestyle="--", alpha=0.5)
ax_bar_zero.set_ylim(-0.25, 0.15)
for i, (name, v) in enumerate(zip(zs_names, zs_rhos)):
    pv = multidomain[name]["pval"]
    offset = 0.012 if v >= 0 else -0.025
    va_align = "bottom" if v >= 0 else "top"
    ax_bar_zero.text(i, v + offset, f"rho = {v:+.3f}\np = {pv:.2e}",
                     ha="center", va=va_align, fontweight="bold", fontsize=10)

fig.suptitle(
    "Figure 6: ChemiSearch — Comprehensive Benchmark & Baseline Results\n"
    "DNA-Based Semantic Search via Differentiable Thermodynamic Sequence Optimization",
    fontweight="bold", fontsize=14, y=1.01,
)
fig_path = os.path.join(WORKSPACE, "figures", "fig6_comprehensive_comparison.png")
fig.savefig(fig_path, bbox_inches="tight")
fig.savefig("./figures/fig6_comprehensive_comparison.png", bbox_inches="tight")
plt.show()
print(f"Figure 6 saved: {fig_path}")


## Section 12 — Cell 12: Qualitative Molecular Case Study
Demonstrates physical hybridization affinity (-DeltaG) discrimination for near-duplicate vs semantically mismatched sentences under true antiparallel alignment.

In [ ]:
# =============================================================================
# Cell 12: Qualitative Molecular Case Study
# =============================================================================

def to_dna(onehot: torch.Tensor) -> str:
    """Decode a (seq_len, 4) one-hot tensor to a 5'->3' nucleotide string."""
    return "".join("ACGT"[i] for i in onehot.argmax(-1).cpu().tolist())

texts = [
    "A patient diagnosed with severe hypertension.",
    "The individual is suffering from very high blood pressure.",
    "The cat is sleeping peacefully on the sofa.",
]

_teacher = SentenceTransformer("all-MiniLM-L6-v2")
embs     = _teacher.encode(texts, convert_to_tensor=True).to(device)

encoder.eval()
with torch.no_grad():
    oh = encoder(embs)   # Deterministic argmax projection

seqs = [to_dna(oh[i]) for i in range(3)]

with torch.no_grad():
    aff_12 = predictor(oh[0:1], oh[1:2]).item()
    aff_13 = predictor(oh[0:1], oh[2:3]).item()

# Analyze position-by-position base matches under antiparallel alignment
wc_map = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
q_dna  = seqs[0]
t1_rc  = "".join(wc_map[b] for b in seqs[1][::-1])
t2_rc  = "".join(wc_map[b] for b in seqs[2][::-1])

m12 = sum(1 for q, t in zip(q_dna, t1_rc) if q == t)
m13 = sum(1 for q, t in zip(q_dna, t2_rc) if q == t)

sep = "=" * 78
print(sep)
print("QUALITATIVE CASE STUDY: DNA HYBRIDIZATION THERMODYNAMICS")
print(sep)
print("[QUERY]")
print(f"  Text : {texts[0]}")
print(f"  DNA  : 5'-{seqs[0]}-3'")
print()
print("[TARGET 1  --  Semantic Near-Duplicate]")
print(f"  Text : {texts[1]}")
print(f"  DNA  : 5'-{seqs[1]}-3'")
print(f"  Antiparallel Watson-Crick Matches : {m12} / 128 ({m12/128*100:.1f}%)")
print(f"  Predicted Hybridisation Affinity (-DeltaG) : {aff_12:.2f}")
print()
print("[TARGET 2  --  Semantic Mismatch]")
print(f"  Text : {texts[2]}")
print(f"  DNA  : 5'-{seqs[2]}-3'")
print(f"  Antiparallel Watson-Crick Matches : {m13} / 128 ({m13/128*100:.1f}%)")
print(f"  Predicted Hybridisation Affinity (-DeltaG) : {aff_13:.2f}")
print()
delta = aff_12 - aff_13
print(f"  Affinity Delta (Match - Mismatch) : {delta:+.2f}")
if delta > 0:
    print("  Outcome : Successfully discriminated: Higher physical affinity for semantic match.")
else:
    print("  Outcome : Non-discriminative or inverted on this individual pair.")
print()
print("  Biophysical Commentary:")
print("  In physical molecular search, duplex stability is governed by collective nearest-neighbor")
print("  enthalpy/entropy and non-linear mismatch penalties. While rank correlation holds across large")
print("  evaluation corpora (e.g. STS-B test rho = +0.073, p = 3.36e-3), individual pair margins depend")
print("  heavily on sequence composition and localized mismatch context.")
print(sep)


## Section 13 — Cell 13: Summary Results Table & Release Packaging
Consolidates all multi-domain results, baseline comparisons, and ablation metrics; packages artifacts into release archive.

In [ ]:
# =============================================================================
# Cell 13: Summary Results Table & Release Packaging
# =============================================================================

print("=" * 66)
print("RESULTS SUMMARY — DNA-Based Semantic Search (ChemiSearch)")
print("=" * 66)

print("\nTable 1: Multi-Domain Semantic Evaluation")
print(f"  {'Dataset':<12}  {'Domain / Split Type':<26}  {'n':>5}  {'Spearman rho':>13}  {'p-value':>12}")
print("  " + "-" * 74)
for name, domain in [
    ("STS-B",   "In-Domain (Held-Out)"),
    ("SICK-R",  "Zero-Shot Commonsense"),
    ("BIOSSES", "Zero-Shot Biomedical"),
    ("STS17",   "Zero-Shot Cross-Lingual"),
]:
    tdd = test_data[name]
    encoder.eval()
    with torch.no_grad():
        e1 = tdd["e1"].to(device)
        e2 = tdd["e2"].to(device)
        ta = predictor(encoder(e1), encoder(e2)).cpu().numpy()
    r, p = stats.spearmanr(tdd["scores"], ta)
    print(f"  {name:<12}  {domain:<26}  {len(tdd['scores']):>5}  {r:>13.4f}  {p:>12.3e}")

print("\nTable 2: Baseline Comparison (STS-B in-domain test)")
print(f"  {'Method':<28}  {'Spearman rho':>13}")
print("  " + "-" * 44)
for k in sorted_base:
    marker = "  <-- ChemiSearch (Ours)" if "Ours" in k else ""
    print(f"  {k:<28}  {baseline_rhos[k]:>13.4f}{marker}")

print("\nTable 3: Ablation Study (STS-B in-domain test)")
print(f"  {'Variant':<38}  {'Spearman rho':>13}")
print("  " + "-" * 54)
for k in abl_order:
    marker = "  <-- Full Model" if "Full" in k else ""
    print(f"  {k:<38}  {ablation_results[k]:>13.4f}{marker}")

# -- Export ------------------------------------------------------------------
export_dir = os.path.join(WORKSPACE, "export")
os.makedirs(export_dir, exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE, "figures"),
                os.path.join(export_dir, "figures"), dirs_exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE, "models"),
                os.path.join(export_dir, "models"),  dirs_exist_ok=True)

# Also ensure ./figures in repo root has all generated figures
shutil.copytree(os.path.join(WORKSPACE, "figures"),
                "./figures", dirs_exist_ok=True)

zip_base = os.path.join(WORKSPACE, "DNA_Semantic_Search_Release")
shutil.make_archive(zip_base, "zip", export_dir)
print(f"\nAll artifacts packaged: {zip_base}.zip")

from IPython.display import FileLink
display(FileLink("dna_search_workspace/DNA_Semantic_Search_Release.zip"))
